In [1]:
import torch
import pandas as pd
import numpy as np
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import ast

/home/aniketj/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ==== Configuration ====
csv_path = "output/mim_trocr_predictions.csv"
image_folder = "Working_dataset/test"
clip_model_name = "openai/clip-vit-base-patch32"
alpha = 0.5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# ==== Load Data ====
df = pd.read_csv(csv_path)
df["predictions"] = df["predictions"].apply(ast.literal_eval)
df["logprobs"] = df["logprobs"].apply(ast.literal_eval)

In [4]:
# ==== Load CLIP ====
clip_model = CLIPModel.from_pretrained(clip_model_name).to(device).eval()
clip_processor = CLIPProcessor.from_pretrained(clip_model_name)

In [5]:
# ==== Process Each Image ====
for idx, row in df.iterrows():
    image_path = f"{image_folder}/{row['image_name']}"
    pred_texts = row["predictions"]
    logprobs = row["logprobs"]

    try:
        image = Image.open(image_path).convert("RGB")
    except Exception as e:
        print(f"Error loading image: {image_path} — {e}")
        continue

    # CLIP Inference
    inputs = clip_processor(text=pred_texts, images=image, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        outputs = clip_model(**inputs)
        clip_probs = outputs.logits_per_image.softmax(dim=1)[0].cpu().tolist()

    # Normalize logprobs
    logprobs_np = np.array(logprobs)
    norm_logprobs = (logprobs_np - logprobs_np.min()) / (logprobs_np.max() - logprobs_np.min() + 1e-8)
    final_scores = alpha * np.array(clip_probs) + (1 - alpha) * norm_logprobs

    # Rerank predictions
    reranked = sorted(
        zip(pred_texts, logprobs, clip_probs, final_scores),
        key=lambda x: x[3], reverse=True
    )

    # ==== Output ====
    print("-------------------------------------------------------------------------------------------")
    print(f"\n Image: {row['image_name']}")
    print(f" Ground Truth         : {row['ground_truth']}\n")

    # Top prediction
    top_pred, top_lp, top_cp, top_score = reranked[0]
    print(f" Top Prediction       : {top_pred}")
    print(f"    LogProb           : {top_lp:.2f}")
    print(f"    CLIP Score        : {top_cp:.4f}")
    print(f"    Combined Score    : {top_score:.4f}")

    # Show all predictions
    print("\n All Predictions:")
    for i, (pred, lp, cp, score) in enumerate(reranked):
        print(f"{i+1}. {pred}")
        print(f"    LogProb: {lp:.2f},  CLIP: {cp:.4f},  Score: {score:.4f}")
    print("-------------------------------------------------------------------------------------------")

-------------------------------------------------------------------------------------------

 Image: Rodrigo_00416_00.png
 Ground Truth         : Garci hernandez de castilla e don Gonçalo gustios el padre

 Top Prediction       : Galaci hernandez de Castilla e don Gonçalo gustos el padre
    LogProb           : -423.23
    CLIP Score        : 0.0867
    Combined Score    : 0.5433

 All Predictions:
1. Galaci hernandez de Castilla e don Gonçalo gustos el padre
    LogProb: -423.23,  CLIP: 0.0867,  Score: 0.5433
2. Galacio herciendo y de Castilla e don Gonçalo gustios el padre
    LogProb: -426.39,  CLIP: 0.0831,  Score: 0.5067
3. Gacoi hernandez de Castilla e don Gonçalo gustios el padre
    LogProb: -461.35,  CLIP: 0.5648,  Score: 0.3615
4. Galacio herci hercios hecha e don Gonçalo gustios el padre
    LogProb: -439.55,  CLIP: 0.0665,  Score: 0.3531
5. Galaci hernandez de Castilla e don Gonçalo gustios el padre
    LogProb: -468.51,  CLIP: 0.1988,  Score: 0.0994
-----------------------

Token indices sequence length is longer than the specified maximum sequence length for this model (196 > 77). Running this sequence through the model will result in indexing errors


-------------------------------------------------------------------------------------------

 Image: Rodrigo_00443_17.png
 Ground Truth         : do Abdalla la touo en su poder quisose llegar a ella como a su

 Top Prediction       : do Abdalla la touo en su poder quiso se llegar a ella como a su
    LogProb           : -353.62
    CLIP Score        : 0.2281
    Combined Score    : 0.6140

 All Predictions:
1. do Abdalla la touo en su poder quiso se llegar a ella como a su
    LogProb: -353.62,  CLIP: 0.2281,  Score: 0.6140
2. do Abdalla la touo en su poder quisose llegar a ella como a su
    LogProb: -396.57,  CLIP: 0.2190,  Score: 0.5009
3. do Abdalla la touo en su poder quiso se lugar a ella como a su
    LogProb: -453.36,  CLIP: 0.1058,  Score: 0.3008
4. do Abdalla la touo en su poder quiso se llegar a ella como a su
    LogProb: -494.88,  CLIP: 0.2281,  Score: 0.2570
5. do Abdalla la touo en su poder quisose llegar a ella como a su
    LogProb: -551.44,  CLIP: 0.2190,  Score: 0.10

RuntimeError: The size of tensor a (196) must match the size of tensor b (77) at non-singleton dimension 1